# Feature Engineering on Weather and Hourly Demand Datasets:

----

# Import Libraries:

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F 
from pyspark.sql.functions import * 
import os

In [ ]:
# Create a spark session (which will run spark jobs)
spark = (
    SparkSession.builder.appName("feature_engineering_weather+demand")
    .config("spark.sql.repl.eagerEval.enabled", True)
    .config("spark.network.timeout", "600s")
    .config("spark.driver.maxResultSize", "2g")
    .config("spark.rpc.askTimeout", "600s")
    .config("spark.driver.memory", "100G")
    .config("spark.executor.memory", "100G")
    .config("spark.sql.parquet.cacheMetadata", "true")
    .config("spark.sql.session.timeZone", "Etc/UTC")
    .config("spark.sql.debug.maxToStringFields", "1000")
    .getOrCreate()
)

# Read Files:

In [ ]:
base_dir = "../data"

Hourly weather dataset:

In [ ]:
hourly_weather_sdf_path = base_dir + '/curated/weather_data/preprocessed_hourly_weather'
hourly_weather_sdf = spark.read.parquet(hourly_weather_sdf_path)
hourly_weather_sdf.show(5)

Hourly demand dataset:

In [ ]:
hourly_demand_sdf_dir = base_dir + '/developed/merged_data/hourly_pickup_demand'
hourly_demand_sdf = spark.read.parquet(hourly_demand_sdf_dir)
hourly_demand_sdf.show(5)

# Aggregated Hourly Demand Dataset:

In [ ]:
hourly_demand_sdf = hourly_demand_sdf.drop('day_type')
hourly_demand_sdf = hourly_demand_sdf.groupBy('pickup_hour').agg(
    F.avg('mean_hourly_demand').alias('avg_hourly_demand')
)
hourly_demand_sdf = hourly_demand_sdf.withColumnRenamed('pickup_hour', 'hour')

hourly_demand_sdf.show(5)

# Merge Two Datasets:

In [ ]:
# Rename the `Hour` column of `hourly_weather_sdf` to `hour` for merging
hourly_weather_sdf = hourly_weather_sdf.withColumnRenamed('Hour', 'hour')

# Merge by `hour`` column
hourly_demand_by_weather = hourly_demand_sdf.join(hourly_weather_sdf, on='hour', how='inner')

hourly_demand_by_weather.show(5)

# Save the Merged Dataset:

In [ ]:
hourly_demand_by_weather_dir = base_dir + '/developed/merged_data'
file_name = 'hourly_demand_by_weather'
hourly_demand_by_weather_path = os.path.join(hourly_demand_by_weather_dir, file_name)
hourly_demand_by_weather.write.mode('overwrite').parquet(hourly_demand_by_weather_path)